In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import LinearSegmentedColormap
#Boundary conditions for instability
# Slightly uneven inlet speed
ymid = (SY - 1) / 2.0             # middle index in y

U_PROF = U_in * (                 # inlet speed at each height y, an array of length SY
    1.0                           # start from the average speed (factor 1)
    + EPS * (np.arange(SY) - ymid) / (SY / 2.0)   # add a straight slope: -EPS at bottom, +EPS at top
)                                 # end of the inlet profile
def set_bnd_u(u):                 # set the boundary values of the horizontal velocity u

    u[0, :] = U_PROF              # left edge (inlet): the uneven inlet profile
    u[-1, :] = u[-2, :]           # right edge (outlet): copy the neighbouring column

    # Bottom fixed wall
    u[:, 0] = 0.0                 # bottom edge: fixed wall, u = 0 (no-slip)

    # Top moving wall
    u[:, -1] = TOP_WALL_SPEED     # top edge: wall moving right at 1.5 times the inlet speed


def set_bnd_v(v):                 # set the boundary values of the vertical velocity v

    v[0, :] = V_in                # left edge (inlet): vertical speed = V_in = 0
    v[-1, :] = v[-2, :]           # right edge (outlet): copy the neighbouring column

    # No vertical flow through either wall
    v[:, 0] = 0.0                 # bottom wall: no flow through it
    v[:, -1] = 0.0                # top wall: no flow through it

In [ ]:
#Strouhal number 
def strouhal(probe):
 
    # Use only the late part of the signal: the second half of the run
    late = probe[len(probe) // 2:]
    # Subtract the mean so the signal oscillates around zero
    late = late - late.mean()
 
    # If the biggest swing is below 0.05, there is no clear shedding
    if np.abs(late).max() < 0.05:
        return None, None
 
    # A crossing is the moment the probe signal passes through zero going upward,
    # and the time between two of them is one shedding period.
    crossings = np.where(
        (late[:-1] < 0)       # this sample is below zero
        & (late[1:] >= 0)     # the next sample is at or above zero
    )[0]                      # keep only the index positions of these crossings
 
    # With fewer than 3 crossings the period would be unreliable
    if len(crossings) < 3:
        return None, None
 
    # Mean number of steps between crossings, times dt, gives the period T
    period = np.diff(crossings).mean() * dt
    # Frequency is one over the period
    frequency = 1.0 / period
 
    # Strouhal number: frequency times circle diameter over inflow speed
    St = frequency * D / U_in
 
    # Return both the Strouhal number and the period
    return St, period